In [34]:
from app.models import Embedding, SubTopic, Bookmark, Document_Entity_Link, Entity, Document
from app.db_connector import get_engine
import app.getters as getter
from app.transformers_util import generate_embeddings, get_util
from sqlalchemy.orm import Session
from sqlalchemy import (
    text,
    and_, or_, select
)

engine = get_engine()
user_id = 'HqAXhad3jrUWmPibnMf1xZczNIq2'


In [61]:
def add_embedding_to_subtopic(embedding: Embedding, 
                             threshold: float = 0.5) -> SubTopic | None:
    """Method that adds an embedding to a subtopic if it matches the cosine similarity threshold."""
    result = None
    with Session(engine) as session:
        ## Query to find the subtopic that matches the embedding
        stmt = text("""
                    SELECT c.subtopic_id, MAX(c.cosine_similarity) as cosine_similarity, json_agg(DISTINCT c.embedding_id) as embedding_ids
                    FROM (SELECT a.emb_id AS embedding_id, a.subtopic_id AS subtopic_id, a.cosine_similarity AS cosine_similarity
                    FROM (SELECT e.id AS emb_id, l.subtopic_id AS subtopic_id, MAX(1 - (e.vector <=> :vector)) AS cosine_similarity 
                            FROM embedding AS e
                            JOIN subtopic_embedding_link l ON l.embedding_id = e.id
                            WHERE e.user_id = :user_id
                            AND l.subtopic_id IS NOT NULL
                            AND e.id != :needle_id
                            GROUP BY 1, 2) a
                    JOIN subtopic_embedding_link b ON a.subtopic_id = b.subtopic_id
                    WHERE a.cosine_similarity >= :threshold
                    ORDER BY a.cosine_similarity DESC
                    LIMIT :limit) c
                    GROUP BY 1
                    """)
        
        matches_subtopic = session.execute(stmt, {"vector": str(embedding.vector.tolist()), 
                                                 "user_id": embedding.user_id, 
                                                 "needle_id": embedding.id, 
                                                 "threshold": threshold, "limit": 20}).fetchall()
        return matches_subtopic




In [63]:
emb = getter.get_embedding_by_id(668)
subtopics = add_embedding_to_subtopic(emb, 0.01)

for subtopic in subtopics:
    note = f"Matching embedding to {emb.id} are {subtopic.embedding_ids} with cosine similarity {subtopic.cosine_similarity}"
    print(subtopic.subtopic_id, subtopic.cosine_similarity, note)

18 0.6524023595335924 Matching embedding to 668 are [665, 666, 667, 672] with cosine similarity 0.6524023595335924
19 0.556352665098252 Matching embedding to 668 are [664, 692] with cosine similarity 0.556352665098252


In [40]:
with Session(engine) as session:
        
        stmt = select(Entity)\
            .join(Document_Entity_Link, Document_Entity_Link.entity_id == Entity.id)\
            .join(Bookmark, Bookmark.document_id == Document_Entity_Link.document_id)\
            .outerjoin(Embedding, Embedding.source_id == Entity.id)\
            .where(Embedding.source_id == None, Bookmark.user_id == user_id)
        
        print(stmt)
        entities = session.scalars(stmt).unique().all()
        
for entity in entities:
    print(entity.name, entity.id, entity.version)

SELECT entity.id, entity.version, entity.name, entity.description, entity.descriptions_bank, entity.source, entity.type, entity.wikidata_id, entity.score, entity.update_at 
FROM entity JOIN document_entity_link ON document_entity_link.entity_id = entity.id JOIN bookmark ON bookmark.document_id = document_entity_link.document_id LEFT OUTER JOIN embedding ON embedding.source_id = entity.id 
WHERE embedding.source_id IS NULL AND bookmark.user_id = :user_id_1


In [ ]:
SELECT entity.id, entity.version, entity.name, entity.description, entity.descriptions_bank, entity.source, entity.type, entity.wikidata_id, entity.score, entity.update_at 
FROM entity JOIN document_entity_link ON document_entity_link.entity_id = entity.id JOIN bookmark ON bookmark.document_id = document_entity_link.document_id LEFT OUTER JOIN embedding ON embedding.source_id = entity.id 
WHERE (embedding.source_id IS NULL OR embedding.version < entity.version) AND bookmark.user_id = :user_id_1